# 006 — нейросеть D/LIVE + уверенные эпизоды H/A из 005
Run All: подготовка признаков, 12 эпох обучения на 4 минутах, проверка последней минуты; новая модель с нуля на всех 5 минутах; полный прогноз. Есть tqdm, progress.json, history.csv, checkpoint после каждой эпохи. Прежние эксперименты не изменяются.

Архитектура 002: Transformer 2×64, 4 головы → GRU(64), теперь **2 выхода LIVE/D**. Обучается заново, не загружает неудачные трёхклассовые веса. Веса классов вычисляются только на обучающем отрезке; среднее взвешенной CE берётся по числу кадров, чтобы веса не сокращались в одноклассовом блоке. Штраф устойчивости 0.05 действует внутри одинаковой истинной метки. Последовательное обучение TBPTT сохранено для сопоставимости с 002; его ограничения остаются.

D при вероятности ≥0.5. Офлайн-фильтр сначала заполняет LIVE-разрывы до 10 кадров, окружённые D, затем удаляет D-эпизоды короче 25 кадров (1 секунды). Никакого переноса через перерыв. Это эвристика длительности, не правило футбола; короткая настоящая остановка может быть потеряна. Порог и длительности фиксированы до оценки, не подбираются по проверке. Их можно менять в первой ячейке конфигурации, но для нового обучения задавайте новый RUN_NAME; текущие значения фиксированы в config.

В живых интервалах применяется 005 с ранее выбранными параметрами. A—неопределённость—A заполняется A; через новый D переходы не протягиваются. Проверочная минута изолирована и для GRU, и для якорей; будущие проверочные данные не участвуют в обучении. Финальная модель на 5 минутах не имеет независимого теста. Публичная проверка уже многократно использовалась в проекте.

Все выходы локальны: local_review/experiments/006_binary_dead_anchors/run_001. Сохранены сырые вероятности D и итоговые маски, диагностические результаты до/после фильтра, финальный submission.csv. Закрытая разметка не читается. Новых библиотек нет.


In [ ]:
from pathlib import Path
import os, sys, json, time, random, hashlib, platform
import numpy as np
import pandas as pd
import torch
from torch import nn
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
sys.path.insert(0, str(ROOT))
from games import GAME, GAMES, asset_paths, half_video_blocks
from starter.baseline import team_of_each_track
from evaluation import possession_metrics as metrics

# CPU is the reproducible default; CUDA can be selected explicitly.
# CPU — воспроизводимый вариант по умолчанию; CUDA можно включить вручную.
DEVICE = "cpu"
RUN_NAME = "run_001"
EPOCHS = 12
SEED = 42
SEQ_LEN = 125                 # 5 seconds / 5 секунд на шаг градиента
SMOOTH_WEIGHT = 0.05
LEARNING_RATE = 3e-4
DEAD_THRESHOLD = 0.5
MIN_DEAD_FRAMES = 25
MAX_LIVE_GAP_FRAMES = 10
RUN_FINAL = True              # Refit on all 5 min / Обучить заново на всех 5 минутах
RUN_FULL_MATCH = True         # Predict after final training / Прогноз после финального обучения
RESUME = True
SMOKE = os.environ.get("POSSESSION_SMOKE") == "1"  # Developer check / Короткая проверка кода
if SMOKE:
    RUN_NAME, EPOCHS, SEQ_LEN = "smoke_test", 1, 32
    RUN_FINAL, RUN_FULL_MATCH = False, False
OUT = ROOT / "local_review/experiments/006_binary_dead_anchors" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "local_review/experiments/002_transformer_gru/features_v1"
CACHE.mkdir(parents=True, exist_ok=True)
PATHS = asset_paths()
N, FPS = GAMES[GAME]["n_video_frames"], GAMES[GAME]["fps"]
LENGTH, WIDTH = GAMES[GAME]["pitch"]
BLOCKS = half_video_blocks(GAME)
LABELS = np.array(["LIVE", "D"])
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True)
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
seed_all()

def save_json(path, value):
    # Atomic replacement prevents half-written status files.
    # Атомарная замена защищает журнал от частичной записи.
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False))
    tmp.replace(path)

def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

config = dict(seed=SEED, epochs=EPOCHS, seq_len=SEQ_LEN, smooth_weight=SMOOTH_WEIGHT,
              learning_rate=LEARNING_RATE, device=DEVICE, d_model=64, heads=4, layers=2,
              gru_hidden=64, dropout=0.1, features_version=1, smoke=SMOKE, target="binary_live_dead",
              weighted_ce=True, dead_threshold=DEAD_THRESHOLD, min_dead_frames=MIN_DEAD_FRAMES, max_live_gap_frames=MAX_LIVE_GAP_FRAMES,
              torch=torch.__version__, numpy=np.__version__, python=platform.python_version())
config_path = OUT / "config.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == config, "Config changed: use a new RUN_NAME"
save_json(config_path, config)
print("Device:", DEVICE, "| Results:", OUT)


## Признаки и бинарные цели

In [ ]:
teams = team_of_each_track(PATHS["atd_meta"])
P = len(teams)
source_signature = dict(raw_sha256=digest(PATHS["atd_raw"]),
                        meta_sha256=digest(PATHS["atd_meta"]), version=2)
manifest = CACHE / "manifest.json"
cache_ok = (manifest.exists() and json.loads(manifest.read_text()) == source_signature
            and (CACHE / "features.npy").exists() and (CACHE / "padding.npy").exists())
if not cache_ok:
    # Missing positions stay NaN until the explicit feature/mask conversion.
    # Пропуски остаются NaN до явного преобразования признаков и масок.
    xy = np.full((N, P + 1, 2), np.nan, dtype=np.float64)
    with open(PATHS["atd_raw"]) as f:
        count = 0
        for i, line in enumerate(tqdm(f, total=N, desc="Read tracking / Чтение")):
            frame, players, ball = line.strip().split(":")
            assert int(frame) == i + 1
            xy[i, :-1] = np.fromstring(players.replace(";", ","), sep=",").reshape(P, 3)[:, :2]
            xy[i, -1] = np.fromstring(ball, sep=",")[:2]
            count += 1
    assert count == N
    seen = np.isfinite(xy).all(axis=-1)
    keeper = np.zeros((N, P), dtype=bool)
    suspect = np.zeros((N, P), dtype=bool)
    lines = np.full((N, 2), np.nan, dtype=np.float64)
    sides = {}
    for half, first, last in BLOCKS:
        anchor = {1: 11185, 2: 113742}[half]
        centers = [np.nanmedian(np.nanmedian(xy[anchor:anchor+250, :P, 0][:, teams==t], axis=1))
                   for t in (0, 1)]
        assert abs(centers[0]-centers[1]) > .05
        sides[half] = [centers[0] < centers[1], centers[1] < centers[0]]
        rows = np.arange(first, last+1)
        for team in (0, 1):
            slots = np.flatnonzero(teams == team)
            x = xy[first:last+1, :P, 0][:, slots].copy()
            visible = seen[first:last+1, :P][:, slots]
            left = sides[half][team]
            fill = np.inf if left else -np.inf
            x[~visible] = fill
            candidate = x.argmin(axis=1) if left else x.argmax(axis=1)
            rr = np.arange(len(rows))
            depth = (x[rr, candidate] if left else 1-x[rr, candidate]) * LENGTH
            eligible = visible.any(axis=1) & (depth >= -1e-10) & (depth <= 16.5 + 1e-10)
            keeper[rows[eligible], slots[candidate[eligible]]] = True
            x[rr[eligible], candidate[eligible]] = fill
            line = x.min(axis=1) if left else x.max(axis=1)
            line[visible.sum(axis=1)-eligible.astype(int) == 0] = np.nan
            lines[rows, team] = line
        for team in (0, 1):
            slots = np.flatnonzero(teams == team)
            x = xy[first:last+1, :P, 0][:, slots]
            line = lines[first:last+1, 1-team, None]
            behind = x < line if sides[half][1-team] else x > line
            suspect[np.ix_(rows, slots)] = seen[first:last+1, :P][:, slots] & np.isfinite(line) & behind
    features = np.lib.format.open_memmap(CACHE / "features.npy", mode="w+", dtype="float32", shape=(N,P+1,11))
    padding = np.lib.format.open_memmap(CACHE / "padding.npy", mode="w+", dtype="bool", shape=(N,P+1))
    for first in tqdm(range(0, N, 2048), desc="Features / Признаки"):
        end = min(first+2048, N)
        present = seen[first:end]
        bseen = present[:, -1]
        raw = xy[first:end]
        f = np.zeros((end-first,P+1,11), dtype=np.float32)
        f[..., :2] = np.nan_to_num(raw, nan=0.)
        # Scale relative displacement by pitch dimensions, not sample statistics.
        # Масштаб смещения фиксирован, статистики выборки не нужны.
        delta = raw - raw[:, -1:, :]
        f[..., 2:4] = np.where((present & bseen[:,None])[...,None], delta, 0.)
        f[..., 4] = bseen[:,None]
        f[:, :P, 5] = teams == 0
        f[:, :P, 6] = teams == 1
        f[:, -1, 7] = 1
        f[:, :P, 8] = keeper[first:end]
        f[:, :P, 9] = suspect[first:end]
        f[..., 10] = present
        features[first:end] = f
        padding[first:end] = ~present
        padding[first:end, -1] = False  # Keep explicit missing-ball token / Оставляем токен мяча
    features.flush(); padding.flush()
    save_json(manifest, source_signature)  # Written last / Записываем только после завершения
    del xy, seen, keeper, suspect, lines, features, padding
features = np.load(CACHE / "features.npy", mmap_mode="r")
padding = np.load(CACHE / "padding.npy", mmap_mode="r")
assert features.shape == (N, P+1, 11)

# Only the PUBLIC truth file is opened. No private truth or proxy labels.
# Читаем только публичную разметку; закрытая разметка не используется.
gt_path = ROOT / "evaluation/gt/BAR-ATH_possession_gt_public.csv"
gt = pd.read_csv(gt_path)
assert np.array_equal(gt.frame.to_numpy(), np.arange(N))
y = np.full(N, -1, dtype=np.int64)
y[gt.label.isin(['H','A']).to_numpy()] = 0
y[gt.label.eq('D').to_numpy()] = 1
labeled = np.flatnonzero(y >= 0)
assert len(labeled) == 300 * FPS and np.all(np.diff(labeled) == 1)
START, END = int(labeled[0]), int(labeled[-1]+1)
SPLIT = START + 240 * FPS
TRAIN_END, VAL_END = (START+64, SPLIT+64) if SMOKE else (SPLIT, END)
assert TRAIN_END <= SPLIT < VAL_END <= END
save_json(OUT / "data_signature.json", dict(**source_signature, public_gt_sha256=digest(gt_path)))
print("Train frames:", TRAIN_END-START, "| Validation:", VAL_END-SPLIT)


## Transformer → GRU с двумя выходами

In [ ]:
class PossessionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(11, 64)
        self.cls = nn.Parameter(torch.zeros(1, 1, 64))
        layer = nn.TransformerEncoderLayer(64, 4, dim_feedforward=128, dropout=.1,
                                           activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(64)
        self.gru = nn.GRU(64, 64, batch_first=True)
        self.head = nn.Linear(64, 2)
        # Distinct initial weights across encoder layers / Разная инициализация слоёв
        for module in self.encoder.layers:
            for p in module.parameters():
                if p.ndim > 1:
                    nn.init.xavier_uniform_(p)

    def forward(self, x, mask, hidden=None):
        # Each frame is a set; the first dimension here is time, not player identity.
        # Каждый кадр — множество; первая ось здесь время, а не игроки.
        tokens = self.input(x)
        cls = self.cls.expand(len(x), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        mask = torch.cat([torch.zeros((len(x),1), dtype=torch.bool, device=x.device), mask], dim=1)
        embedding = self.norm(self.encoder(tokens, src_key_padding_mask=mask)[:, 0])
        temporal, hidden = self.gru(embedding.unsqueeze(0), hidden)
        return self.head(temporal.squeeze(0)), hidden

def tensor_block(first, end):
    # Copy memory-map slices so PyTorch never mutates read-only arrays.
    # Копия среза защищает исходный массив от изменений.
    return (torch.tensor(np.array(features[first:end]), device=DEVICE),
            torch.tensor(np.array(padding[first:end]), device=DEVICE))

def objective(logits, target, previous_p=None, previous_y=None, class_weights=None):
    ce = nn.functional.cross_entropy(logits, target, weight=class_weights, reduction="none").mean()
    probabilities = logits.softmax(-1)
    pp, yy = probabilities, target
    if previous_p is not None:
        pp = torch.cat([previous_p[None], pp])
        yy = torch.cat([previous_y[None], yy])
    same = yy[1:] == yy[:-1]
    variation = (pp[1:] - pp[:-1]).abs().sum(-1)
    smooth = variation[same].mean() if same.any() else probabilities.sum() * 0
    return ce + SMOOTH_WEIGHT*smooth, ce, smooth

@torch.no_grad()
def predict(model, first, end, show=True):
    model.eval()
    hidden = None
    result = np.empty(end-first, dtype=np.int64)
    for a in tqdm(range(first, end, SEQ_LEN), desc="Predict / Прогноз", disable=not show, leave=False):
        b = min(a+SEQ_LEN, end)
        logits, hidden = model(*tensor_block(a,b), hidden)
        result[a-first:b-first] = logits.argmax(-1).cpu().numpy()
        if show:
            save_json(OUT / "progress.json", dict(status="inference", first_frame=first,
                      frames_done=b-first, frames_total=end-first))
    return result

def score(truth, prediction):
    # Binary F1 computed explicitly; LIVE is not the H label of the competition.
    # Бинарный F1: LIVE нельзя путать с классом H соревнования.
    result={}
    f1s=[]
    for k,name in enumerate(['live','dead']):
        tp=int(((truth==k)&(prediction==k)).sum())
        fp=int(((truth!=k)&(prediction==k)).sum())
        fn=int(((truth==k)&(prediction!=k)).sum())
        precision=tp/max(tp+fp,1); recall=tp/max(tp+fn,1)
        f1=2*tp/max(2*tp+fp+fn,1)
        result.update({name+'_precision':precision,name+'_recall':recall,name+'F1':f1})
        f1s.append(f1)
    result['macroF1']=float(np.mean(f1s))
    return result

@torch.no_grad()
def dead_probabilities(model,first,end):
    model.eval();hidden=None;result=np.empty(end-first,dtype=np.float32)
    for i in tqdm(range(first,end,SEQ_LEN),desc='P(D) / Вероятность остановки'):
        stop=min(i+SEQ_LEN,end)
        logits,hidden=model(*tensor_block(i,stop),hidden)
        result[i-first:stop-first]=logits.softmax(-1)[:,1].cpu().numpy()
        save_json(OUT/'progress.json',dict(stage='inference',first_frame=first,frames_done=stop-first,frames_total=end-first))
    return result

# Structural checks: missing objects, permutation invariance, finite gradients.
# Проверяем пропуски, независимость от порядка игроков и конечность градиентов.
probe = PossessionNet().to(DEVICE).eval()
x, mask = tensor_block(START, START+8)
with torch.no_grad():
    a, _ = probe(x, mask)
    perm = torch.randperm(P+1, device=DEVICE)
    b, _ = probe(x[:,perm], mask[:,perm])
    torch.testing.assert_close(a, b, atol=2e-6, rtol=2e-5)
    empty_mask = torch.ones_like(mask); empty_mask[:,-1] = False
    empty_logits, _ = probe(torch.zeros_like(x), empty_mask)
    assert torch.isfinite(empty_logits).all()
# Temporal chunking preserves inference; later inputs cannot affect earlier answers.
# Разбиение на блоки сохраняет прогноз; будущие кадры не меняют прошлый ответ.
with torch.no_grad():
    prefix, h = probe(x[:4], mask[:4])
    suffix, _ = probe(x[4:], mask[4:], h)
    torch.testing.assert_close(a, torch.cat([prefix,suffix]), atol=2e-6, rtol=2e-5)
    changing = torch.tensor([[5.,0.,0.],[0.,5.,0.]], device=DEVICE)
    _, _, transition_penalty = objective(changing, torch.tensor([0,1], device=DEVICE))
    _, _, stable_penalty = objective(changing, torch.tensor([0,0], device=DEVICE))
    assert transition_penalty.item() == 0 and stable_penalty.item() > 0
probe.train()
logits, _ = probe(x,mask)
loss, _, _ = objective(logits, torch.tensor(y[START:START+8], device=DEVICE))
loss.backward()
assert all(torch.isfinite(p.grad).all() for p in probe.parameters() if p.grad is not None)
assert any(p.grad is not None and p.grad.abs().sum()>0 for p in probe.encoder.parameters())
print("Checks passed | Parameters:", sum(p.numel() for p in probe.parameters()))
del probe


## Обучение и контроль последней минуты

In [ ]:
def train(stage, first, end, validate=False):
    folder = OUT / stage
    folder.mkdir(exist_ok=True)
    checkpoint = folder / "last.pt"
    seed_all()
    model = PossessionNet().to(DEVICE)
    # Class weights are fitted only on this stage's training frames.
    # Веса классов вычисляются только по обучающим кадрам данного этапа.
    counts=np.bincount(y[first:end],minlength=2)
    assert np.all(counts>0) or SMOKE
    class_weights=torch.tensor((end-first)/(2*np.maximum(counts,1)),dtype=torch.float32,device=DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=.01)
    history, initial_epoch = [], 0
    if RESUME and checkpoint.exists():
        # Only load our own locally created checkpoint / Только собственный локальный checkpoint
        state = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
        assert state["config"] == config and state["data_signature"] == json.loads((OUT / "data_signature.json").read_text())
        assert state["span"] == [first,end]
        model.load_state_dict(state["model"]); optimizer.load_state_dict(state["optimizer"])
        history, initial_epoch = state["history"], state["epoch"]
        torch.set_rng_state(state["torch_rng"].cpu())
        if DEVICE.startswith("cuda"):
            torch.cuda.set_rng_state_all(state["cuda_rng"])
        print(stage, "resuming after epoch", initial_epoch)
    for epoch in tqdm(range(initial_epoch, EPOCHS), desc=stage + " / Эпохи"):
        model.train()
        hidden, previous_p, previous_y = None, None, None
        total_loss, total_ce, total_smooth, frames = 0., 0., 0., 0
        started = time.time()
        blocks = tqdm(range(first, end, SEQ_LEN), desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
        for a in blocks:
            b = min(a+SEQ_LEN, end)
            optimizer.zero_grad(set_to_none=True)
            target = torch.tensor(y[a:b], device=DEVICE)
            logits, hidden = model(*tensor_block(a,b), hidden)
            loss, ce, smooth = objective(logits, target, previous_p, previous_y, class_weights)
            assert torch.isfinite(loss), "Non-finite training loss"
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            # Carry memory, not the graph / Переносим память, но не граф градиентов
            hidden = hidden.detach()
            previous_p, previous_y = logits[-1].softmax(-1).detach(), target[-1].detach()
            size = b-a; frames += size
            total_loss += loss.item()*size; total_ce += ce.item()*size; total_smooth += smooth.item()*size
            blocks.set_postfix(loss=f"{total_loss/frames:.4f}")
            save_json(OUT / "progress.json", dict(stage=stage, status="training", epoch=epoch+1,
                      epochs=EPOCHS, frames_done=frames, frames_total=end-first, loss=total_loss/frames))
        row = dict(epoch=epoch+1, loss=total_loss/frames, ce=total_ce/frames,
                   smooth=total_smooth/frames, seconds=time.time()-started)
        if validate:
            prediction = predict(model, SPLIT, VAL_END, show=False)
            row.update({"val_"+k:v for k,v in score(y[SPLIT:VAL_END], prediction).items()})
        history.append(row)
        state = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), epoch=epoch+1,
                     history=history, config=config, span=[first,end], torch_rng=torch.get_rng_state(),
                     cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.startswith("cuda") else [],
                     data_signature=json.loads((OUT / "data_signature.json").read_text()))
        temporary = checkpoint.with_suffix(".tmp")
        torch.save(state, temporary); temporary.replace(checkpoint)
        pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
        tqdm.write(f"{stage}: epoch {epoch+1}/{EPOCHS}, loss={row['loss']:.4f}" +
                   (f", val deadF1={row['val_deadF1']:.4f}" if validate else ""))
    pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
    save_json(OUT / "progress.json", dict(stage=stage, status="completed", epochs=EPOCHS))
    return model, pd.DataFrame(history)

diagnostic_model, diagnostic_history = train("diagnostic_4min", START, TRAIN_END, validate=True)

validation_probability=dead_probabilities(diagnostic_model,SPLIT,VAL_END)
np.save(OUT/'validation_dead_probability.npy',validation_probability)
if RUN_FINAL:
    final_model,final_history=train('final_5min',START,END)


## Фильтр длительности D и прогноз полного матча

In [ ]:
def spans(values):
    if not len(values):return []
    starts=np.r_[0,np.flatnonzero(values[1:]!=values[:-1])+1]
    return list(zip(starts,np.r_[starts[1:],len(values)]))

def filter_dead(probability,minimum=MIN_DEAD_FRAMES,max_gap=MAX_LIVE_GAP_FRAMES,threshold=DEAD_THRESHOLD):
    # Offline duration filtering, not interpolation of unobserved ball coordinates.
    # Офлайн-фильтр длительности состояний, а не интерполяция координат мяча.
    raw=probability>=threshold;joined=raw.copy()
    # Fill only bounded LIVE gaps, never the unbounded interval edges.
    # Заполняем только короткий LIVE между двумя D, не края интервала.
    for a,b in spans(raw):
        if not raw[a] and a>0 and b<len(raw) and b-a<=max_gap:
            joined[a:b]=True
    result=joined.copy()
    # Apply minimum AFTER joining; no output D episode can be shorter than minimum.
    # Минимум проверяем ПОСЛЕ объединения: коротких D на выходе не остаётся.
    for a,b in spans(joined):
        if joined[a] and b-a<minimum:result[a:b]=False
    assert all(b-a>=minimum for a,b in spans(result) if result[a])
    return result

def check_duration_filter():
    def f(p): return filter_dead(p,minimum=25,max_gap=10,threshold=.5)
    assert not f(np.r_[np.zeros(30),np.ones(10),np.zeros(30)]).any()
    assert f(np.r_[np.ones(25),np.zeros(10),np.ones(25)]).all()
    assert not f(np.r_[np.zeros(5),np.ones(25)])[:5].any()
    assert f(np.ones(25)).all()
    assert not f(np.ones(24)).any()
check_duration_filter()
validation_dead=filter_dead(validation_probability)
save_json(OUT/'binary_validation_metrics.json',dict(raw=score(y[SPLIT:VAL_END],(validation_probability>=DEAD_THRESHOLD).astype(int)),
                filtered=score(y[SPLIT:VAL_END],validation_dead.astype(int))))
probability=np.full(N,np.nan,dtype=np.float32)
raw_dead=np.zeros(N,bool);filtered_dead=np.zeros(N,bool)
if RUN_FULL_MATCH and RUN_FINAL:
    for half,first,last in BLOCKS:
        probability[first:last+1]=dead_probabilities(final_model,first,last+1)
        raw_dead[first:last+1]=probability[first:last+1]>=DEAD_THRESHOLD
        filtered_dead[first:last+1]=filter_dead(probability[first:last+1])
    np.save(OUT/'full_dead_probability.npy',probability)
    np.save(OUT/'full_dead_mask.npy',filtered_dead)
print('Duration filter verified')


In [ ]:
# Reuse raw geometry for the frozen H/A algorithm 005.
# Исходная геометрия для фиксированного алгоритма H/A из 005.
CFG={'box_width_m':5.}
PITCH=np.array(GAMES[GAME]['pitch'],dtype=np.float64)
# Load only ATD positions; raw speed units are inconsistent in the source documentation.
# Читаем только координаты ATD: единицы готовой скорости неоднозначны.
track_teams = team_of_each_track(PATHS["atd_meta"])
players = np.full((N, len(track_teams), 2), np.nan, dtype=np.float64)
ball = np.full((N, 2), np.nan, dtype=np.float64)
read_count = 0
with open(PATHS["atd_raw"]) as handle:
    for i, line in enumerate(tqdm(handle,total=N,desc="Tracking / Трекинг")):
        frame_id, player_text, ball_text = line.strip().split(":")
        assert int(frame_id) == i + 1, "ATD/video alignment mismatch"
        player_values = np.fromstring(player_text.replace(";", ","), sep=",").reshape(-1, 3)
        assert len(player_values) == len(track_teams)
        players[i] = player_values[:, :2] * PITCH
        ball[i] = np.fromstring(ball_text, sep=",")[:2] * PITCH
        read_count += 1
assert read_count == N
ball_seen = np.isfinite(ball).all(axis=1)
players_seen = np.isfinite(players).all(axis=2)

# The window is an axis-aligned SQUARE, not a radius-2.5 m circle.
# Окно — КВАДРАТ вдоль осей поля, не круг радиусом 2.5 м.
def box_votes(player_xy, ball_xy, team_ids, width_m):
    inside = (np.isfinite(player_xy).all(axis=2)
              & np.isfinite(ball_xy).all(axis=1)[:, None]
              & (np.abs(player_xy - ball_xy[:, None, :]) <= width_m / 2).all(axis=2))
    home_count = (inside & (team_ids == 0)).sum(axis=1)
    away_count = (inside & (team_ids == 1)).sum(axis=1)
    # +1 means H, -1 means A, 0 means abstention for this frame.
    # +1 — H, -1 — A, 0 — нет голоса в этом кадре.
    votes = np.sign(home_count - away_count).astype(np.int8)
    return home_count, away_count, votes

home_count, away_count, frame_votes = box_votes(players, ball, track_teams, CFG["box_width_m"])
# A narrow synthetic check verifies corners, ties, and missing-ball handling.
# Проверяем на искусственных точках углы квадрата, ничью и отсутствие мяча.
test_players = np.array([[[2.5, 2.5], [2.6, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]]])
test_ball = np.array([[0., 0.], [0., 0.], [np.nan, np.nan]])
assert box_votes(test_players, test_ball, np.array([0, 1]), 5)[2].tolist() == [1, 0, 0]



In [ ]:
# Compute only aggregate geometric support; no track identity is assumed.
# Геометрические свидетельства без предположения о постоянной личности слота.
distance=np.linalg.norm(players-ball[:,None,:],axis=2)
distance[~players_seen|~ball_seen[:,None]]=np.inf
nearest=np.column_stack([distance[:,track_teams==t].min(axis=1) for t in (0,1)])
team_counts=np.column_stack([players_seen[:,track_teams==t].sum(axis=1) for t in (0,1)])
reliable=ball_seen&(team_counts>=3).all(axis=1)
del distance

def runs_of(values):
    starts=np.r_[0,np.flatnonzero(values[1:]!=values[:-1])+1]
    ends=np.r_[starts[1:],len(values)]
    return zip(starts,ends)

def smooth_evidence(values,window):
    # Offline centered smoothing, strictly inside the current live interval.
    # Офлайн-сглаживание строго внутри текущего игрового интервала.
    ids=np.arange(len(values));left=np.maximum(0,ids-window//2)
    right=np.minimum(len(values),ids+window//2+1)
    sums=np.vstack([np.zeros((1,2)),np.cumsum(values,axis=0)])
    return (sums[right]-sums[left])/(right-left)[:,None]

def choose_boundary(evidence,start,stop,old,new,method):
    # Boundary k means old team before k, new team from k onward.
    # Граница k: старая команда до k, новая начиная с k.
    middle=(start+stop)/2
    if method=='midpoint': return int(np.floor(middle))
    segment=evidence[start:stop]
    if not len(segment) or not np.any(segment): return int(np.floor(middle))
    if method=='consistent':
        old_prefix=np.r_[0.,np.cumsum(segment[:,old])]
        new_prefix=np.r_[0.,np.cumsum(segment[:,new])]
        objective=old_prefix+new_prefix[-1]-new_prefix
        candidates=np.flatnonzero(np.isclose(objective,objective.max(),atol=1e-12,rtol=0))+start
        return int(candidates[np.argmin(abs(candidates-middle))])
    if method=='weighted':
        # Rising support for the new team plus a small contest-proximity term.
        # Рост преимущества новой команды плюс небольшой вес близости обеих команд.
        contrast=evidence[:,new]-evidence[:,old]
        candidates=np.arange(start,stop+1)
        now=np.minimum(candidates,len(evidence)-1)
        prev=np.maximum(candidates-1,0)
        rise=np.maximum(contrast[now]-contrast[prev],0.)
        contested=np.minimum(evidence[now,old],evidence[now,new])
        weight=rise+.1*contested
        if weight.sum()<=1e-12:return int(np.floor(middle))
        return int(np.clip(np.floor(np.dot(candidates,weight)/weight.sum()+.5),start,stop))
    raise ValueError(method)

def fill_live(raw,evidence,min_frames,method,edge_frames=50):
    # 0=H, 1=A, 3=X. Confident runs are accepted retrospectively in their entirety.
    # Уверенную серию принимаем целиком после проверки её длины; метод офлайн.
    prediction=np.full(len(raw),3,dtype=np.int8); anchors=[]; gaps=[]
    if not len(raw):return prediction,anchors,gaps
    for a,b in runs_of(raw):
        if raw[a]!=3 and b-a>=min_frames:
            anchors.append((int(a),int(b),int(raw[a])))
            prediction[a:b]=raw[a]
    if not anchors:return prediction,anchors,gaps
    a,b,label=anchors[0];prediction[max(0,a-edge_frames):a]=label
    a,b,label=anchors[-1];prediction[b:min(len(raw),b+edge_frames)]=label
    for left,right in zip(anchors,anchors[1:]):
        _,start,old=left;stop,_,new=right
        if old==new:
            # The main rule: same confident team on both sides implies NO turnover.
            # Главное правило: одинаковые команды по краям означают НЕТ перехода.
            prediction[start:stop]=old
            boundary=None
        else:
            boundary=choose_boundary(evidence,start,stop,old,new,method)
            prediction[start:boundary]=old;prediction[boundary:stop]=new
        gaps.append(dict(start=start,stop=stop,old=old,new=new,boundary=boundary))
    return prediction,anchors,gaps

def predict_interval(cfg,first,stop,collect=False):
    # Crucially, train and validation calls cannot see each other's anchors/evidence.
    # Train и validation не видят якоря и свидетельства соседнего раздела.
    result=np.full(stop-first,3,dtype=np.int8);details=[];anchor_count=0;anchor_frames=0
    for half,hfirst,hlast in BLOCKS:
        a=max(first,hfirst);b=min(stop,hlast+1)
        if a>=b:continue
        states=dead[a:b]
        for local_start,local_stop in runs_of(states):
            x=a+local_start;y=a+local_stop
            if states[local_start]:result[x-first:y-first]=2;continue
            d=nearest[x:y];ok=reliable[x:y]
            raw=np.full(y-x,3,dtype=np.int8)
            for team in (0,1):
                confident=ok&(d[:,team]<=cfg['radius_m'])&(d[:,1-team]>=cfg['radius_m']+cfg['opponent_margin_m'])
                raw[confident]=team
            # Gaussian distance support; these scores are NOT calibrated probabilities.
            # Гауссовы оценки близости НЕ являются калиброванными вероятностями.
            evidence=np.exp(-.5*(d/max(cfg['radius_m'],1.))**2)
            evidence[~ok]=0.
            evidence=smooth_evidence(evidence,cfg['evidence_window_frames'])
            pred,anchors,gaps=fill_live(raw,evidence,cfg['min_confident_frames'],cfg['boundary_method'])
            result[x-first:y-first]=pred
            anchor_count+=len(anchors);anchor_frames+=sum(j-i for i,j,t in anchors)
            if collect:
                for gap in gaps:
                    details.append(dict(start_frame=x+gap['start'],end_frame_exclusive=x+gap['stop'],
                        left_team='HA'[gap['old']],right_team='HA'[gap['new']],
                        boundary_frame=None if gap['boundary'] is None else x+gap['boundary'],
                        rule='same_team_no_turnover' if gap['old']==gap['new'] else cfg['boundary_method']))
    return np.array(['H','A','D','X'])[result],dict(anchor_runs=anchor_count,anchor_frames=anchor_frames),details

# Behavioral tests: same-team bridge, exact confident labels, all missing, and changepoint.
# Проверки: одинаковая команда по краям, неизменность якорей, пропуски и граница.
for method in ('midpoint','weighted','consistent'):
    raw=np.array([1,1,3,3,3,1,1]);p,_,_=fill_live(raw,np.zeros((7,2)),2,method)
    assert np.all(p==1)
    raw=np.array([0,0,3,3,1,1]);p,_,_=fill_live(raw,np.zeros((6,2)),2,method)
    assert np.all(p[:2]==0) and np.all(p[-2:]==1)
    assert np.count_nonzero(p[1:]!=p[:-1])==1
    p,_,_=fill_live(np.full(7,3),np.zeros((7,2)),2,method)
    assert np.all(p==3)
e=np.array([[1.,0.],[1.,0.],[1.,0.],[0.,1.],[0.,1.],[0.,1.]])
assert choose_boundary(e,2,4,0,1,'consistent')==3
print('Anchor rules verified')


## Комбинация с H/A из 005

In [ ]:
anchor_cfg=json.loads((ROOT/'sol/005_confident_anchors/selected_training_config.json').read_text())
save_json(OUT/'anchor_config.json',anchor_cfg)
# All H/A parameters remain frozen at experiment 005's training-selected values.
# Все параметры H/A фиксированы по эксперименту 005; нового подбора нет.
assert not SMOKE, 'Use the full run for end-to-end predictions'
dead=np.zeros(N,bool);dead[SPLIT:VAL_END]=validation_dead
validation_labels,val_anchor_stats,_=predict_interval(anchor_cfg,SPLIT,VAL_END)
# Raw-gate control isolates the effect of duration filtering, without reselecting parameters.
# Контроль с сырым D выделяет эффект фильтра, не меняя параметры.
dead[SPLIT:VAL_END]=validation_probability>=DEAD_THRESHOLD
validation_raw_gate,_,_=predict_interval(anchor_cfg,SPLIT,VAL_END)

def possession_score(pred,first,stop):
    true=metrics.label_codes(gt.label.to_numpy()[first:stop]);codes=metrics.label_codes(pred)
    cm=metrics.scored_confusion(metrics.confusion(true,codes));runs=metrics.valid_runs(true)
    spell=metrics.spell_scores(true,metrics.fill_abstentions(codes,runs),runs)
    return dict(macroF1=metrics.macro_f1(cm),spellF1=spell['mean_f1'],X_fraction=float(np.mean(codes==3)),
        **{name+'_F1':metrics.per_class_prf(cm,k)[2] for k,name in enumerate(['H','A','D'])})
pd.DataFrame({'frame':np.arange(SPLIT,VAL_END),'label':validation_labels}).to_csv(OUT/'validation_predictions.csv',index=False)
report=dict(validation=possession_score(validation_labels,SPLIT,VAL_END),
            validation_raw_gate=possession_score(validation_raw_gate,SPLIT,VAL_END),validation_anchors=val_anchor_stats)
if RUN_FULL_MATCH and RUN_FINAL:
    dead=filtered_dead
    labels,anchor_stats,gaps=predict_interval(anchor_cfg,0,N,collect=True)
    assert np.array_equal(labels=='D',filtered_dead)
    for g in gaps:
        a,b=g['start_frame'],g['end_frame_exclusive']
        assert not dead[a:b].any()
        if g['left_team']==g['right_team']:assert np.all(labels[a:b]==g['left_team'])
    result=pd.DataFrame({'frame':np.arange(N),'label':labels})
    error=metrics.validate_submission(result,N);assert error is None,error
    result.to_csv(OUT/'submission.csv',index=False)
    pd.DataFrame(gaps).to_csv(OUT/'gap_diagnostics.csv',index=False)
    report.update(public_all=possession_score(labels[START:END],START,END),full_anchors=anchor_stats)
    report['binary_public_training']={
        'raw':score(y[START:END],raw_dead[START:END].astype(int)),
        'filtered':score(y[START:END],filtered_dead[START:END].astype(int))}
    report['duration_filter']={'min_dead_frames':MIN_DEAD_FRAMES,'max_live_gap_frames':MAX_LIVE_GAP_FRAMES,
        'frames_changed':int(np.count_nonzero(raw_dead!=filtered_dead)),
        'raw_dead_frames':int(raw_dead.sum()),'filtered_dead_frames':int(filtered_dead.sum())}
    save_json(OUT/'progress.json',dict(status='completed',output='submission.csv',sha256=digest(OUT/'submission.csv')))
save_json(OUT/'metrics.json',report)
print(json.dumps(report,indent=2))


## Веса для прогноза и локальный отчёт

In [ ]:
# Save inference-only weights without optimizer, labels, or training histories.
# Сохраняем только веса для прогноза: без оптимизатора, меток и истории обучения.
if RUN_FINAL and (OUT/'final_5min/last.pt').exists():
    trained=torch.load(OUT/'final_5min/last.pt',map_location='cpu',weights_only=False)
    torch.save(trained['model'],OUT/'final_model_weights.pt')
    save_json(OUT/'inference_config.json',dict(architecture='Transformer(2 layers,64 dim,4 heads)+GRU(64)',
        input_features=11,outputs=['LIVE','D'],threshold=DEAD_THRESHOLD,
        min_dead_frames=MIN_DEAD_FRAMES,max_live_gap_frames=MAX_LIVE_GAP_FRAMES,
        anchor_config=json.loads((OUT/'anchor_config.json').read_text()),
        training_config=config))
    del trained
r=json.loads((OUT/'metrics.json').read_text())
binary=json.loads((OUT/'binary_validation_metrics.json').read_text())
text=['# Эксперимент 006 — бинарная сеть D/LIVE и H/A из 005','',
      'Проверка: сеть обучена на первых четырёх минутах, последняя минута изолирована для GRU и якорей.',
      'Финальная версия: другая сеть обучена на всех пяти минутах. Оценка на них является оценкой на данных обучения.',
      '', '| Отрезок | spellF1 | macroF1 | H F1 | A F1 | D F1 |', '|---|---:|---:|---:|---:|---:|']
for key in ['validation','public_all']:
    if key in r:
        m=r[key];text.append(f"| {key} | {m['spellF1']:.4f} | {m['macroF1']:.4f} | {m['H_F1']:.4f} | {m['A_F1']:.4f} | {m['D_F1']:.4f} |")
text+=['','Бинарная диагностика до/после фильтра: '+json.dumps(binary,ensure_ascii=False),
       '', 'Фильтр: заполняются LIVE-разрывы до 10 кадров между D, затем удаляются D-эпизоды короче 25 кадров. Это фиксированная эвристика, не факт о правилах футбола.',
       'На проверочной минуте D не предсказан ни разу; фильтр не способен восстановить отсутствующее свидетельство остановки. Хорошие тренировочные метрики не означают переноса на остальной матч.',
       'Параметры после проверки не изменялись. Сохранены вероятности D, сырые/сглаженные диагностические метрики и полный submission.']
(OUT/'REVIEW_RU.md').write_text('\n'.join(text)+'\n')
